# Chunking Engine

## Objective

Convert standardized Document Objects into standardized Chunk Objects.

This engine is responsible only for splitting documents.

It does not:

- Read PDFs
- Generate embeddings
- Store vectors
- Query LLMs

Input

Data/processed/

↓

document.json

Output

Data/chunks/

document_id_chunks.json

The output of this notebook becomes the input for the Embedding Engine.


In [1]:
!pip install -q tqdm
from google.colab import drive

drive.mount("/content/drive")
import json
from pathlib import Path

print("Environment Ready")

Mounted at /content/drive
Environment Ready


In [2]:
ROOT = Path("/content/drive/MyDrive/MicroBrain")

PROCESSED = ROOT / "Data" / "processed"

CHUNKS = ROOT / "Data" / "chunks"

METADATA = ROOT / "Data" / "metadata"

In [3]:
documents = sorted(
    PROCESSED.glob("*.json")
)

print(len(documents))

1


In [4]:
for doc in documents:

    print(doc.name)

a60be88b-ec91-429e-8ab4-c1bb43cdf6e6.json


In [5]:
document_path = documents[0]

with open(document_path, "r", encoding="utf-8") as file:

    document = json.load(file)

# Step 6 - Standard Chunk Object

Instead of representing chunks as plain strings, each chunk is stored
as a structured object.

Every downstream engine (Embedding, Retrieval, Prompting, Generation)
will consume this object.

This makes the pipeline modular and extensible.

In [11]:
document_text = document["content"]["text"]

print(type(document_text))

print()

print(len(document_text))

<class 'str'>

9837


In [17]:
CHUNK_SIZE = 500

OVERLAP = 100

In [18]:
chunk_objects = []

In [20]:
import uuid

chunk_objects = []

STEP = CHUNK_SIZE - OVERLAP

for start in range(0, len(document_text), STEP):

    end = start + CHUNK_SIZE

    chunk_text = document_text[start:end]

    if len(chunk_text) == 0:
        continue

    chunk = {

        "chunk_id": str(uuid.uuid4()),

        "document_id": document["id"],

        "chunk_index": len(chunk_objects),

        "start_character": start,

        "end_character": min(end, len(document_text)),

        "metadata": {

            "chunk_size": len(chunk_text),

            "overlap": OVERLAP,

            "strategy": "fixed_character"

        },

        "content": {

            "text": chunk_text

        }

    }

    chunk_objects.append(chunk)

# Step 7 - Save Chunk Objects

Persist standardized chunk objects.

Future engines will load these chunks directly rather than
recomputing them from documents.

In [24]:
chunk_file = CHUNKS / f"{document['id']}_chunks.json"

print(chunk_file)

/content/drive/MyDrive/MicroBrain/Data/chunks/a60be88b-ec91-429e-8ab4-c1bb43cdf6e6_chunks.json


In [25]:
import json

with open(chunk_file, "w", encoding="utf-8") as file:

    json.dump(
        chunk_objects,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Chunks Saved")

Chunks Saved


# Step 8 - Chunk Registry

Maintain metadata about generated chunk collections.

In [28]:
chunk_registry = METADATA / "chunks.json"

if chunk_registry.exists():

    with open(chunk_registry, "r") as file:

        registry = json.load(file)

else:

    registry = []

In [29]:
registry.append({

    "document_id": document["id"],

    "chunk_file": chunk_file.name,

    "chunk_count": len(chunk_objects),

    "strategy": "fixed_character",

    "chunk_size": CHUNK_SIZE,

    "overlap": OVERLAP

})

In [30]:
with open(chunk_registry, "w") as file:

    json.dump(
        registry,
        file,
        indent=4
    )

print(chunk_registry)

/content/drive/MyDrive/MicroBrain/Data/metadata/chunks.json
